In [0]:
# Databricks notebook - Open-Meteo ingestion class (sem API key)
# Se precisar:
# %pip install requests pandas

import json
import math
import time
import requests
from typing import Any, Dict, List, Optional
from datetime import datetime, timedelta, timezone

import pandas as pd


class OpenMeteoDataFrameExtractor:
    """
    Classe para ingestão Open-Meteo -> Spark DataFrames (Databricks)
    - Sem autenticação (API pública Open-Meteo)
    - Geocoding + Forecast + Historical Weather + Historical Forecast
    - Gera DataFrames prontos para dashboard / ML / forecasting
    """

    URLS = {
        "forecast": "https://api.open-meteo.com/v1/forecast",
        "archive": "https://archive-api.open-meteo.com/v1/archive",
        "historical_forecast": "https://historical-forecast-api.open-meteo.com/v1/forecast",
        "geocoding_search": "https://geocoding-api.open-meteo.com/v1/search",
        "geocoding_get": "https://geocoding-api.open-meteo.com/v1/get",
    }

    def __init__(self, spark, timeout_sec: int = 60, max_retries: int = 4):
        self.spark = spark
        self.timeout_sec = timeout_sec
        self.max_retries = max_retries
        self.session = requests.Session()

    # ============================================================
    # Helpers
    # ============================================================
    def _request_json(self, url: str, params: Optional[Dict[str, Any]] = None) -> Any:
        last_err = None
        for attempt in range(1, self.max_retries + 1):
            try:
                r = self.session.get(url, params=params, timeout=self.timeout_sec)

                if r.status_code == 429:
                    wait_s = min(2 ** attempt, 20)
                    time.sleep(wait_s)
                    continue

                r.raise_for_status()
                return r.json()

            except Exception as e:
                last_err = e
                if attempt < self.max_retries:
                    wait_s = min(2 ** attempt, 20)
                    time.sleep(wait_s)
                else:
                    raise last_err

    @staticmethod
    def _safe_json(v):
        if isinstance(v, (dict, list)):
            return json.dumps(v, ensure_ascii=False)
        return v

    @staticmethod
    def _flatten_dict(d: Dict[str, Any], parent_key: str = "", sep: str = "__") -> Dict[str, Any]:
        out = {}
        for k, v in d.items():
            nk = f"{parent_key}{sep}{k}" if parent_key else str(k)
            if isinstance(v, dict):
                out.update(OpenMeteoDataFrameExtractor._flatten_dict(v, nk, sep))
            elif isinstance(v, list):
                out[nk] = json.dumps(v, ensure_ascii=False)
                out[f"{nk}{sep}count"] = len(v)
            else:
                out[nk] = v
        return out

    @staticmethod
    def _to_spark_df(spark, data):
        """
        Converte list[dict], dict ou pandas.DataFrame para Spark DataFrame
        evitando incompatibilidades de Arrow (Int32/UInt32/pandas nullable dtypes).
        """
        if data is None:
            return spark.createDataFrame([])

        # Normaliza para pandas DataFrame
        if isinstance(data, pd.DataFrame):
            pdf = data.copy()
        elif isinstance(data, list):
            if len(data) == 0:
                return spark.createDataFrame([])
            pdf = pd.DataFrame(data)
        elif isinstance(data, dict):
            pdf = pd.DataFrame([data])
        else:
            pdf = pd.DataFrame([{"value": data}])

        if pdf.empty:
            return spark.createDataFrame([])

        # Converte NaN/NaT/pd.NA para None
        pdf = pdf.where(pd.notnull(pdf), None)

        # Converte datetimes para Python datetime (Spark infere melhor)
        for c in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[c]):
                pdf[c] = pdf[c].apply(lambda x: x.to_pydatetime() if x is not None else None)

        # MUITO IMPORTANTE:
        # transforma em list[dict] para evitar caminho pandas->Arrow (que quebra com Int32/UInt32)
        records = pdf.to_dict(orient="records")

        return spark.createDataFrame(records)

    @staticmethod
    def _filter_geocoding_results(
        results: List[Dict[str, Any]],
        admin1_exact: Optional[str] = None,
        country_exact: Optional[str] = None,
        feature_code_exact: Optional[str] = None,
        prefer_highest_population: bool = False
    ) -> List[Dict[str, Any]]:
        out = results

        if admin1_exact:
            out = [
                r for r in out
                if str(r.get("admin1", "")).strip().lower() == admin1_exact.strip().lower()
            ]

        if country_exact:
            out = [
                r for r in out
                if str(r.get("country", "")).strip().lower() == country_exact.strip().lower()
            ]

        if feature_code_exact:
            out = [
                r for r in out
                if str(r.get("feature_code", "")).strip().upper() == feature_code_exact.strip().upper()
            ]

        if prefer_highest_population and out:
            def _pop(x):
                p = x.get("population")
                try:
                    return int(p) if p is not None else -1
                except Exception:
                    return -1
            out = sorted(out, key=_pop, reverse=True)

        return out

    # ============================================================
    # Método genérico (qualquer endpoint JSON -> DataFrame)
    # ============================================================
    def extract_json_df(
        self,
        url: str,
        params: Optional[Dict[str, Any]] = None,
        record_path: Optional[str] = None,
        include_raw_payload: bool = False
    ):
        payload = self._request_json(url, params=params)

        def get_by_path(obj, path):
            if not path:
                return obj
            cur = obj
            for part in path.split("."):
                if isinstance(cur, dict):
                    cur = cur.get(part)
                else:
                    return None
            return cur

        target = get_by_path(payload, record_path)
        rows = []

        if isinstance(target, list):
            for i, item in enumerate(target):
                row = self._flatten_dict(item) if isinstance(item, dict) else {"value": item}
                row["_record_index"] = i
                row["_source_url"] = url
                row["_request_params_json"] = json.dumps(params or {}, ensure_ascii=False)
                if include_raw_payload:
                    row["_raw_payload_json"] = json.dumps(payload, ensure_ascii=False)
                rows.append(row)

        elif isinstance(target, dict):
            row = self._flatten_dict(target)
            row["_source_url"] = url
            row["_request_params_json"] = json.dumps(params or {}, ensure_ascii=False)
            if include_raw_payload:
                row["_raw_payload_json"] = json.dumps(payload, ensure_ascii=False)
            rows.append(row)

        else:
            rows.append({
                "value": self._safe_json(target),
                "_source_url": url,
                "_request_params_json": json.dumps(params or {}, ensure_ascii=False),
                "_raw_payload_json": json.dumps(payload, ensure_ascii=False) if include_raw_payload else None
            })

        return self._to_spark_df(self.spark, rows)

    # ============================================================
    # Geocoding
    # ============================================================
    def geocode_df(
        self,
        city_name: str,
        country_code: Optional[str] = None,
        language: str = "pt",
        count: int = 10,
        admin1_exact: Optional[str] = None,
        country_exact: Optional[str] = None,
        feature_code_exact: Optional[str] = None,
        prefer_highest_population: bool = False
    ):
        params = {
            "name": city_name,
            "count": count,
            "language": language,
            "format": "json"
        }
        if country_code:
            params["countryCode"] = country_code

        payload = self._request_json(self.URLS["geocoding_search"], params=params)
        results = payload.get("results", []) if isinstance(payload, dict) else []

        results = self._filter_geocoding_results(
            results=results,
            admin1_exact=admin1_exact,
            country_exact=country_exact,
            feature_code_exact=feature_code_exact,
            prefer_highest_population=prefer_highest_population
        )

        rows = []
        for i, r in enumerate(results):
            row = self._flatten_dict(r)
            row["_record_index"] = i
            row["_search_city"] = city_name
            row["_source"] = "geocoding_search"
            rows.append(row)

        return self._to_spark_df(self.spark, rows)

    def geocode_first(
        self,
        city_name: str,
        country_code: Optional[str] = None,
        language: str = "pt",
        admin1_exact: Optional[str] = None,
        country_exact: Optional[str] = None,
        feature_code_exact: Optional[str] = None,
        prefer_highest_population: bool = True
    ) -> Dict[str, Any]:
        params = {
            "name": city_name,
            "count": 20,
            "language": language,
            "format": "json"
        }
        if country_code:
            params["countryCode"] = country_code

        payload = self._request_json(self.URLS["geocoding_search"], params=params)
        results = payload.get("results", []) if isinstance(payload, dict) else []

        if not results:
            raise ValueError(f"Nenhuma localização encontrada para '{city_name}'")

        results = self._filter_geocoding_results(
            results=results,
            admin1_exact=admin1_exact,
            country_exact=country_exact,
            feature_code_exact=feature_code_exact,
            prefer_highest_population=prefer_highest_population
        )

        if not results:
            raise ValueError(
                f"Localização encontrada para '{city_name}', mas nenhuma bateu com "
                f"admin1='{admin1_exact}', country='{country_exact}', feature_code='{feature_code_exact}'."
            )

        return results[0]

    # ============================================================
    # Normalização de timeseries Open-Meteo
    # ============================================================
    @staticmethod
    def _timeseries_block_to_rows(
        block_name: str,
        block_data: Dict[str, Any],
        units_data: Optional[Dict[str, Any]],
        meta: Dict[str, Any]
    ) -> List[Dict[str, Any]]:
        if not isinstance(block_data, dict):
            return []

        # current é objeto único
        if block_name == "current":
            row = {}
            for k, v in block_data.items():
                row[k] = OpenMeteoDataFrameExtractor._safe_json(v)
            if units_data:
                for k, v in units_data.items():
                    row[f"{k}__unit"] = v
            row.update(meta)
            row["_dataset"] = "current"
            return [row]

        # hourly / daily => arrays paralelos
        if "time" not in block_data:
            return []

        time_values = block_data.get("time", [])
        n = len(time_values)
        rows = []

        for i in range(n):
            row = {"time": time_values[i]}
            for k, arr in block_data.items():
                if k == "time":
                    continue
                if isinstance(arr, list):
                    row[k] = arr[i] if i < len(arr) else None
                else:
                    row[k] = OpenMeteoDataFrameExtractor._safe_json(arr)

            if units_data:
                for k, v in units_data.items():
                    row[f"{k}__unit"] = v

            row.update(meta)
            row["_dataset"] = block_name
            rows.append(row)

        return rows

    def _weather_payload_to_dfs(self, payload: Dict[str, Any], source_name: str, request_params: Dict[str, Any]):
        if not isinstance(payload, dict):
            raise ValueError("Payload inesperado do Open-Meteo (esperado dict JSON)")

        meta_keys = [
            "latitude", "longitude", "generationtime_ms", "utc_offset_seconds",
            "timezone", "timezone_abbreviation", "elevation"
        ]
        meta = {k: payload.get(k) for k in meta_keys}
        meta["_source"] = source_name
        meta["_request_params_json"] = json.dumps(request_params or {}, ensure_ascii=False)
        meta["_extracted_at_utc"] = datetime.now(timezone.utc).isoformat()

        out = {"meta_df": self._to_spark_df(self.spark, [meta])}

        if "current" in payload:
            current_rows = self._timeseries_block_to_rows(
                "current",
                payload.get("current", {}),
                payload.get("current_units"),
                meta
            )
            out["current_df"] = self._to_spark_df(self.spark, current_rows)

        if "hourly" in payload:
            hourly_rows = self._timeseries_block_to_rows(
                "hourly",
                payload.get("hourly", {}),
                payload.get("hourly_units"),
                meta
            )
            out["hourly_df"] = self._to_spark_df(self.spark, hourly_rows)

        if "daily" in payload:
            daily_rows = self._timeseries_block_to_rows(
                "daily",
                payload.get("daily", {}),
                payload.get("daily_units"),
                meta
            )
            out["daily_df"] = self._to_spark_df(self.spark, daily_rows)

        return out

    # ============================================================
    # Endpoints de clima
    # ============================================================
    def forecast_dfs(
        self,
        latitude: float,
        longitude: float,
        timezone_str: str = "auto",
        hourly: Optional[List[str]] = None,
        daily: Optional[List[str]] = None,
        current: Optional[List[str]] = None,
        forecast_days: Optional[int] = 7,
        past_days: Optional[int] = None
    ) -> Dict[str, Any]:
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "timezone": timezone_str
        }
        if hourly:
            params["hourly"] = ",".join(hourly)
        if daily:
            params["daily"] = ",".join(daily)
        if current:
            params["current"] = ",".join(current)
        if forecast_days is not None:
            params["forecast_days"] = forecast_days
        if past_days is not None:
            params["past_days"] = past_days

        payload = self._request_json(self.URLS["forecast"], params=params)
        return self._weather_payload_to_dfs(payload, "forecast", params)

    def archive_dfs(
        self,
        latitude: float,
        longitude: float,
        start_date: str,
        end_date: str,
        timezone_str: str = "auto",
        hourly: Optional[List[str]] = None,
        daily: Optional[List[str]] = None
    ) -> Dict[str, Any]:
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "start_date": start_date,   # YYYY-MM-DD
            "end_date": end_date,       # YYYY-MM-DD
            "timezone": timezone_str
        }
        if hourly:
            params["hourly"] = ",".join(hourly)
        if daily:
            params["daily"] = ",".join(daily)

        payload = self._request_json(self.URLS["archive"], params=params)
        return self._weather_payload_to_dfs(payload, "archive", params)

    def historical_forecast_dfs(
        self,
        latitude: float,
        longitude: float,
        start_date: str,
        end_date: str,
        timezone_str: str = "auto",
        hourly: Optional[List[str]] = None,
        daily: Optional[List[str]] = None
    ) -> Dict[str, Any]:
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "start_date": start_date,
            "end_date": end_date,
            "timezone": timezone_str
        }
        if hourly:
            params["hourly"] = ",".join(hourly)
        if daily:
            params["daily"] = ",".join(daily)

        payload = self._request_json(self.URLS["historical_forecast"], params=params)
        return self._weather_payload_to_dfs(payload, "historical_forecast", params)

    # ============================================================
    # Feature engineering (daily) para ML / forecasting
    # ============================================================
    def build_ml_daily_features_from_daily_spark(self, daily_spark_df):
        if daily_spark_df is None:
            return self._to_spark_df(self.spark, [])

        pdf = daily_spark_df.toPandas()
        if pdf.empty or "time" not in pdf.columns:
            return self._to_spark_df(self.spark, [])

        pdf = pdf.copy()
        pdf["time"] = pd.to_datetime(pdf["time"], errors="coerce")
        pdf = pdf.sort_values("time").reset_index(drop=True)

        expected_cols = [
            "temperature_2m_max",
            "temperature_2m_min",
            "apparent_temperature_max",
            "apparent_temperature_min",
            "precipitation_sum",
            "rain_sum",
            "wind_speed_10m_max",
            "weather_code"
        ]
        for c in expected_cols:
            if c not in pdf.columns:
                pdf[c] = pd.NA

        # Datas/sazonalidade
        pdf["date"] = pdf["time"].dt.date.astype(str)
        pdf["year"] = pdf["time"].dt.year
        pdf["month"] = pdf["time"].dt.month
        pdf["day"] = pdf["time"].dt.day
        pdf["day_of_week"] = pdf["time"].dt.dayofweek
        pdf["day_of_year"] = pdf["time"].dt.dayofyear
        # isocalendar pode retornar nullable; convertemos depois
        iso_week = pdf["time"].dt.isocalendar().week
        pdf["week_of_year"] = pd.to_numeric(iso_week, errors="coerce").astype("float64")

        pdf["sin_doy"] = (2 * math.pi * pdf["day_of_year"] / 365.25).apply(lambda x: math.sin(x) if pd.notnull(x) else None)
        pdf["cos_doy"] = (2 * math.pi * pdf["day_of_year"] / 365.25).apply(lambda x: math.cos(x) if pd.notnull(x) else None)

        # Base
        tmax = pd.to_numeric(pdf["temperature_2m_max"], errors="coerce")
        tmin = pd.to_numeric(pdf["temperature_2m_min"], errors="coerce")
        psum = pd.to_numeric(pdf["precipitation_sum"], errors="coerce")

        pdf["temp_range"] = tmax - tmin
        pdf["is_rainy_day"] = (psum.fillna(0) > 0).astype(int)

        # Lags/rollings
        lag_cols = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "wind_speed_10m_max"]
        for c in lag_cols:
            s = pd.to_numeric(pdf[c], errors="coerce")
            pdf[f"{c}_lag1"] = s.shift(1)
            pdf[f"{c}_lag2"] = s.shift(2)
            pdf[f"{c}_lag3"] = s.shift(3)
            pdf[f"{c}_lag7"] = s.shift(7)
            pdf[f"{c}_roll3_mean"] = s.shift(1).rolling(3).mean()
            pdf[f"{c}_roll7_mean"] = s.shift(1).rolling(7).mean()
            pdf[f"{c}_roll14_mean"] = s.shift(1).rolling(14).mean()
            pdf[f"{c}_roll7_std"] = s.shift(1).rolling(7).std()

        # Targets
        pdf["target_temp_max_tplus1"] = tmax.shift(-1)
        pdf["target_precip_sum_tplus1"] = psum.shift(-1)
        pdf["target_rain_binary_tplus1"] = (pdf["target_precip_sum_tplus1"].fillna(0) > 0).astype(int)

        keep_meta = [c for c in ["latitude", "longitude", "timezone", "timezone_abbreviation", "_source"] if c in pdf.columns]

        feature_cols = [
            "date", "time",
            "year", "month", "day", "day_of_week", "day_of_year", "week_of_year",
            "sin_doy", "cos_doy",
            "temperature_2m_max", "temperature_2m_min",
            "apparent_temperature_max", "apparent_temperature_min",
            "precipitation_sum", "rain_sum", "wind_speed_10m_max", "weather_code",
            "temp_range", "is_rainy_day",
        ] + [c for c in pdf.columns if ("_lag" in c or "_roll" in c)] + [
            "target_temp_max_tplus1",
            "target_precip_sum_tplus1",
            "target_rain_binary_tplus1",
        ] + keep_meta

        # dedupe preservando ordem
        seen = set()
        feature_cols = [c for c in feature_cols if c in pdf.columns and not (c in seen or seen.add(c))]

        out = pdf[feature_cols].copy()

        # remove última linha sem target
        out_model = out.dropna(subset=["target_temp_max_tplus1"]).reset_index(drop=True)

        return self._to_spark_df(self.spark, out_model)

    # ============================================================
    # Pipeline convenience: cidade -> vários DataFrames
    # ============================================================
    def build_weather_dataframes_for_city(
        self,
        city_name: str,
        country_code: Optional[str] = "BR",
        admin1_exact: Optional[str] = None,            # ex.: "Rio Grande do Sul"
        country_exact: Optional[str] = "Brasil",       # útil para pt-BR
        feature_code_exact: Optional[str] = "PPLA",    # capital administrativa
        history_start_date: Optional[str] = None,
        history_end_date: Optional[str] = None,
        forecast_days: int = 7
    ) -> Dict[str, Any]:
        """
        Retorna dict com DataFrames:
          - geocode_df, location_meta_df
          - forecast_meta/current/hourly/daily
          - history_meta/hourly/daily
          - ml_daily_features_df
        """

        # 1) Geocoding (com filtros para homônimos)
        geo_first = self.geocode_first(
            city_name=city_name,
            country_code=country_code,
            language="pt",
            admin1_exact=admin1_exact,
            country_exact=country_exact,
            feature_code_exact=feature_code_exact,
            prefer_highest_population=True
        )

        geocode_df = self.geocode_df(
            city_name=city_name,
            country_code=country_code,
            language="pt",
            count=20,
            admin1_exact=admin1_exact,
            country_exact=country_exact,
            feature_code_exact=feature_code_exact,
            prefer_highest_population=True
        )

        lat = float(geo_first["latitude"])
        lon = float(geo_first["longitude"])
        tz = geo_first.get("timezone", "auto")

        # 2) Datas default do histórico
        if history_end_date is None:
            history_end_date = datetime.now().strftime("%Y-%m-%d")
        if history_start_date is None:
            history_start_date = (datetime.now() - timedelta(days=180)).strftime("%Y-%m-%d")

        # 3) Forecast
        forecast_bundle = self.forecast_dfs(
            latitude=lat,
            longitude=lon,
            timezone_str=tz,
            hourly=[
                "temperature_2m",
                "relative_humidity_2m",
                "precipitation_probability",
                "precipitation",
                "weather_code",
                "wind_speed_10m",
                "surface_pressure",
                "cloud_cover"
            ],
            daily=[
                "weather_code",
                "temperature_2m_max",
                "temperature_2m_min",
                "apparent_temperature_max",
                "apparent_temperature_min",
                "precipitation_sum",
                "precipitation_probability_max",
                "wind_speed_10m_max",
                "sunrise",
                "sunset"
            ],
            current=[
                "temperature_2m",
                "relative_humidity_2m",
                "precipitation",
                "weather_code",
                "wind_speed_10m"
            ],
            forecast_days=forecast_days,
            past_days=2
        )

        # 4) Histórico (archive)
        history_bundle = self.archive_dfs(
            latitude=lat,
            longitude=lon,
            start_date=history_start_date,
            end_date=history_end_date,
            timezone_str=tz,
            hourly=[
                "temperature_2m",
                "relative_humidity_2m",
                "precipitation",
                "weather_code",
                "wind_speed_10m",
                "surface_pressure",
                "cloud_cover"
            ],
            daily=[
                "weather_code",
                "temperature_2m_max",
                "temperature_2m_min",
                "apparent_temperature_max",
                "apparent_temperature_min",
                "precipitation_sum",
                "rain_sum",
                "wind_speed_10m_max"
            ]
        )

        # 5) Local meta
        location_meta_row = {
            "city_name_input": city_name,
            "country_code_filter": country_code,
            "admin1_filter": admin1_exact,
            "country_filter": country_exact,
            "feature_code_filter": feature_code_exact,
            "resolved_name": geo_first.get("name"),
            "resolved_country": geo_first.get("country"),
            "resolved_admin1": geo_first.get("admin1"),
            "resolved_admin2": geo_first.get("admin2"),
            "latitude": lat,
            "longitude": lon,
            "elevation": geo_first.get("elevation"),
            "timezone": tz,
            "population": geo_first.get("population"),
            "feature_code": geo_first.get("feature_code"),
        }
        location_meta_df = self._to_spark_df(self.spark, [location_meta_row])

        # 6) Features para ML
        ml_daily_features_df = self.build_ml_daily_features_from_daily_spark(history_bundle.get("daily_df"))

        return {
            "geocode_df": geocode_df,
            "location_meta_df": location_meta_df,

            "forecast_meta_df": forecast_bundle.get("meta_df"),
            "forecast_current_df": forecast_bundle.get("current_df"),
            "forecast_hourly_df": forecast_bundle.get("hourly_df"),
            "forecast_daily_df": forecast_bundle.get("daily_df"),

            "history_meta_df": history_bundle.get("meta_df"),
            "history_hourly_df": history_bundle.get("hourly_df"),
            "history_daily_df": history_bundle.get("daily_df"),

            "ml_daily_features_df": ml_daily_features_df
        }


# ============================================================
# EXEMPLO DE USO - PORTO ALEGRE / RIO GRANDE DO SUL
# ============================================================
extractor = OpenMeteoDataFrameExtractor(spark=spark)

dfs = extractor.build_weather_dataframes_for_city(
    city_name="Porto Alegre",
    country_code="BR",
    admin1_exact="Rio Grande do Sul",   # garante RS
    country_exact="Brasil",
    feature_code_exact="PPLA",          # capital (opcional, mas ajuda)
    history_start_date=None,            # default: últimos 180 dias
    history_end_date=None,              # default: hoje
    forecast_days=7
)

# ---- Verificação da localização resolvida (deve vir Porto Alegre / RS)
display(dfs["geocode_df"])
display(dfs["location_meta_df"])

# ---- Forecast (para dashboard / app)
display(dfs["forecast_current_df"])
display(dfs["forecast_hourly_df"])
display(dfs["forecast_daily_df"])

# ---- Histórico (para EDA / treino)
display(dfs["history_hourly_df"])
display(dfs["history_daily_df"])

# ---- Features para ML/forecasting
display(dfs["ml_daily_features_df"])


# ============================================================
# EXEMPLO EXTRA - Historical Forecast (backtesting forecast vs actual)
# ============================================================
geo_pdf = dfs["location_meta_df"].toPandas()
lat = float(geo_pdf.loc[0, "latitude"])
lon = float(geo_pdf.loc[0, "longitude"])
tz = str(geo_pdf.loc[0, "timezone"])

hist_fcst = extractor.historical_forecast_dfs(
    latitude=lat,
    longitude=lon,
    start_date=(datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d"),
    end_date=datetime.now().strftime("%Y-%m-%d"),
    timezone_str=tz,
    hourly=["temperature_2m", "precipitation_probability", "precipitation", "wind_speed_10m"],
    daily=["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "weather_code"]
)

display(hist_fcst["hourly_df"])
display(hist_fcst["daily_df"])